# Agente LangGraph con búsqueda web (Tavily)

Todo en un solo notebook: claves, herramientas, grafo, `build_search_agent()` y `run_agent()`.

## 1. Claves
Carga el `.env` (el `.env` manda con `override=True`) y pide las claves si faltan.

In [ ]:
import os
import getpass
from dotenv import find_dotenv, load_dotenv


def load_keys():
    """Carga las claves del .env y las pide por pantalla si faltan."""
    load_dotenv(find_dotenv(), override=True)

    anthropic_key = os.environ.get("ANTHROPIC_API_KEY", "").strip()
    if not anthropic_key.startswith("sk-ant-"):
        anthropic_key = getpass.getpass("Anthropic API key (sk-ant-...): ").strip()
    os.environ["ANTHROPIC_API_KEY"] = anthropic_key

    if not os.environ.get("TAVILY_API_KEY"):
        os.environ["TAVILY_API_KEY"] = getpass.getpass("Tavily API key:\n").strip()


load_keys()
print("Anthropic key empieza por:", os.environ["ANTHROPIC_API_KEY"][:7])

## 2. Imports, estado y herramientas
El estado es la lista de mensajes (que se van **añadiendo**) y un contador de llamadas al modelo.

In [ ]:
import operator
from typing import Literal
from typing_extensions import Annotated, TypedDict

from langchain.chat_models import init_chat_model
from langchain.messages import AnyMessage, HumanMessage, SystemMessage, ToolMessage
from langchain.tools import tool
from langchain_tavily import TavilySearch
from langgraph.graph import END, START, StateGraph


class MessagesState(TypedDict):
    messages: Annotated[list[AnyMessage], operator.add]
    llm_calls: int


@tool
def multiply(a: int, b: int) -> int:
    """Multiply `a` and `b`.

    Args:
        a: First int
        b: Second int
    """
    return a * b

## 3. Función 1: `build_search_agent()`
Crea el modelo, las herramientas y el grafo `START → llm_call → (tool_node → llm_call) o END`.

In [ ]:
def build_search_agent():
    """Crea y compila el agente (modelo + herramientas + grafo)."""
    model = init_chat_model("claude-haiku-4-5-20251001", temperature=0)

    search_tool = TavilySearch(max_results=5, topic="general")

    tools = [multiply, search_tool]
    tools_by_name = {t.name: t for t in tools}
    model_with_tools = model.bind_tools(tools)

    system_prompt = SystemMessage(
        content=(
            "You are a helpful assistant. Use the search tool to look up "
            "up-to-date information on the web, and the multiply tool for "
            "multiplications. Answer in the same language as the user."
        )
    )

    # Nodo: el modelo decide si responde o pide una herramienta
    def llm_call(state: dict):
        response = model_with_tools.invoke([system_prompt] + state["messages"])
        return {
            "messages": [response],
            "llm_calls": state.get("llm_calls", 0) + 1,
        }

    # Nodo: ejecuta las herramientas que pidió el modelo
    def tool_node(state: dict):
        result = []
        for tool_call in state["messages"][-1].tool_calls:
            selected_tool = tools_by_name[tool_call["name"]]
            observation = selected_tool.invoke(tool_call["args"])
            result.append(
                ToolMessage(content=str(observation), tool_call_id=tool_call["id"])
            )
        return {"messages": result}

    # Decisión: ¿seguimos con una herramienta o terminamos?
    def should_continue(state: MessagesState) -> Literal["tool_node", END]:
        if state["messages"][-1].tool_calls:
            return "tool_node"
        return END

    # Grafo
    agent_builder = StateGraph(MessagesState)
    agent_builder.add_node("llm_call", llm_call)
    agent_builder.add_node("tool_node", tool_node)
    agent_builder.add_edge(START, "llm_call")
    agent_builder.add_conditional_edges("llm_call", should_continue, ["tool_node", END])
    agent_builder.add_edge("tool_node", "llm_call")

    return agent_builder.compile()

## 4. Función 2: `run_agent()`
Lanza el agente con una pregunta, muestra la conversación y devuelve la respuesta final.

In [ ]:
def run_agent(agent, question: str, verbose: bool = True) -> str:
    """Lanza el agente con una pregunta y devuelve la respuesta final."""
    messages = [HumanMessage(content=question)]
    result = agent.invoke({"messages": messages})

    if verbose:
        for m in result["messages"]:
            m.pretty_print()

    final = result["messages"][-1].content
    if isinstance(final, list):  # a veces viene como lista de bloques
        final = "".join(b.get("text", "") for b in final if isinstance(b, dict))
    return final

## 5. Construir el agente (una sola vez) y ver el grafo

In [ ]:
from IPython.display import Image, display

agent = build_search_agent()
display(Image(agent.get_graph(xray=True).draw_mermaid_png()))

## 6. Pruebas

In [ ]:
# Usa la herramienta multiply
respuesta = run_agent(agent, "Multiply 3 and 4.")

In [ ]:
# Usa la herramienta de búsqueda web (Tavily)
respuesta = run_agent(agent, "¿Qué es LangGraph? Búscalo en la web.")
print("\nRESPUESTA FINAL:\n", respuesta)